# Basic Usage Example - Menstrual Cycle Analysis

This notebook demonstrates the basic usage of the menstrual_cycle_analysis package for analyzing sleep and physiological data in relation to menstrual cycles.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Import our analysis package
from menstrual_cycle_analysis import DataProcessor, SleepAnalyzer, CycleAnalyzer

## 1. Create Sample Data

First, let's create some sample data to demonstrate the package functionality.

In [ ]:
# Create sample data for demonstration
np.random.seed(42)

# Generate 90 days of data
dates = pd.date_range(start='2024-01-01', periods=90, freq='D')

# Create sample sleep data with some cycle-related patterns
sample_data = pd.DataFrame({
    'date': dates,
    'sleep_duration': np.random.normal(7.5, 1.0, 90),  # hours
    'sleep_efficiency': np.random.normal(85, 10, 90),   # percentage
    'rem_sleep': np.random.normal(1.8, 0.3, 90),        # hours
    'deep_sleep': np.random.normal(1.2, 0.2, 90),       # hours
    'heart_rate': np.random.normal(65, 8, 90),           # bpm
    'body_temperature': np.random.normal(98.6, 0.5, 90) # fahrenheit
})

# Ensure positive values
sample_data['sleep_duration'] = np.clip(sample_data['sleep_duration'], 4, 12)
sample_data['sleep_efficiency'] = np.clip(sample_data['sleep_efficiency'], 60, 100)
sample_data['rem_sleep'] = np.clip(sample_data['rem_sleep'], 0.5, 3)
sample_data['deep_sleep'] = np.clip(sample_data['deep_sleep'], 0.3, 2.5)

print(f"Created sample dataset with {len(sample_data)} days of data")
sample_data.head()

## 2. Data Processing

Let's use the DataProcessor to clean and process our data.

In [ ]:
# Initialize the data processor
processor = DataProcessor(use_r=False)  # Set to True if you have rpy2 installed

# Load the data (in real use, you'd load from a file)
# processor.load_data('your_data_file.csv')
processor.data = sample_data.copy()

# Get summary statistics
summary = processor.get_summary_stats()
print("Dataset Summary:")
print(f"Shape: {summary['shape']}")
print(f"Missing values: {summary['missing_values']}")

## 3. Sleep Analysis

Now let's analyze sleep patterns using the SleepAnalyzer.

In [ ]:
# Initialize sleep analyzer
sleep_analyzer = SleepAnalyzer(use_r=False)

# Load sleep data
sleep_analyzer.load_sleep_data(sample_data)

# Calculate basic sleep metrics
sleep_metrics = sleep_analyzer.calculate_sleep_metrics()
print("Sleep Metrics:")
for metric, value in sleep_metrics.items():
    print(f"{metric}: {value:.2f}")

## 4. Cycle Analysis

Let's analyze menstrual cycle patterns and identify cycle phases.

In [ ]:
# Initialize cycle analyzer
cycle_analyzer = CycleAnalyzer(use_r=False)

# Load cycle data
cycle_analyzer.load_cycle_data(sample_data)

# Define some sample menstruation dates (every ~28 days)
menstruation_dates = [
    datetime(2024, 1, 1),
    datetime(2024, 1, 29),
    datetime(2024, 2, 26),
    datetime(2024, 3, 25)
]

# Identify cycle phases
cycle_phases = cycle_analyzer.identify_cycle_phases(menstruation_dates)
print("Cycle phase distribution:")
print(cycle_phases.value_counts())

## 5. Sleep Analysis by Cycle Phase

Now let's analyze how sleep patterns vary across different cycle phases.

In [ ]:
# Analyze sleep by cycle phase
sleep_by_phase = sleep_analyzer.analyze_sleep_by_cycle_phase(cycle_phases)

print("Sleep Duration by Cycle Phase:")
for phase, metrics in sleep_by_phase.items():
    if 'mean_sleep_duration' in metrics:
        print(f"{phase}: {metrics['mean_sleep_duration']:.2f} ± {metrics['std_sleep_duration']:.2f} hours")

## 6. Visualization

Let's create some visualizations to explore the data.

In [ ]:
# Prepare data for visualization
plot_data = sample_data.copy()
plot_data['cycle_phase'] = cycle_phases

# Create subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Sleep duration over time
axes[0, 0].plot(plot_data['date'], plot_data['sleep_duration'])
axes[0, 0].set_title('Sleep Duration Over Time')
axes[0, 0].set_ylabel('Hours')
axes[0, 0].tick_params(axis='x', rotation=45)

# Sleep duration by cycle phase
sns.boxplot(data=plot_data, x='cycle_phase', y='sleep_duration', ax=axes[0, 1])
axes[0, 1].set_title('Sleep Duration by Cycle Phase')
axes[0, 1].set_ylabel('Hours')

# Sleep efficiency over time
axes[1, 0].plot(plot_data['date'], plot_data['sleep_efficiency'])
axes[1, 0].set_title('Sleep Efficiency Over Time')
axes[1, 0].set_ylabel('Percentage')
axes[1, 0].tick_params(axis='x', rotation=45)

# Heart rate by cycle phase
sns.boxplot(data=plot_data, x='cycle_phase', y='heart_rate', ax=axes[1, 1])
axes[1, 1].set_title('Heart Rate by Cycle Phase')
axes[1, 1].set_ylabel('BPM')

plt.tight_layout()
plt.show()

## 7. Statistical Analysis

Let's perform some statistical comparisons between cycle phases.

In [ ]:
# Statistical comparison of sleep duration across phases
try:
    stats_result = sleep_analyzer.compare_phases_statistical(
        cycle_phases, 
        sleep_metric='sleep_duration'
    )
    print("Statistical Analysis Results:")
    print(f"Test: {stats_result['test']}")
    if 'f_statistic' in stats_result:
        print(f"F-statistic: {stats_result['f_statistic']:.4f}")
        print(f"P-value: {stats_result['p_value']:.4f}")
    elif 'phase_means' in stats_result:
        print("Phase means:")
        for phase, mean in stats_result['phase_means'].items():
            print(f"  {phase}: {mean:.2f}")
except Exception as e:
    print(f"Statistical analysis error: {e}")

## 8. Cycle Variability Analysis

Finally, let's analyze cycle variability metrics.

In [ ]:
# Calculate cycle variability
variability = cycle_analyzer.calculate_cycle_variability(menstruation_dates)

print("Cycle Variability Metrics:")
for metric, value in variability.items():
    print(f"{metric}: {value:.2f}")

## Conclusion

This notebook demonstrated the basic functionality of the menstrual_cycle_analysis package:

1. **Data Processing**: Loading and cleaning physiological data
2. **Sleep Analysis**: Calculating sleep metrics and analyzing patterns
3. **Cycle Analysis**: Identifying cycle phases and analyzing variability
4. **Statistical Analysis**: Comparing metrics across cycle phases
5. **Visualization**: Creating plots to explore the data

The package provides a foundation for analyzing menstrual cycle physiology and sleep data from wearable devices, with optional R integration for advanced statistical analyses.